# Document Classification Comparison

This notebook runs all three classification methods and compares their results.

**Note:** If you encounter errors about missing fields, restart the kernel to ensure the latest code is loaded.

In [ ]:
# Install the document_classification module in editable mode
import sys
import subprocess
from pathlib import Path

# Get project root (notebooks -> document_classification -> src -> NEW)
project_root = Path.cwd().parent.parent.parent

# Install in editable mode
subprocess.run([
    sys.executable, "-m", "pip", "install", "-e", 
    str(project_root / "src" / "document_classification"),
    "-q"
], check=True)

print("✓ Module installed successfully")

In [ ]:
# Import required libraries
import json
import sys
from pathlib import Path
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import importlib

# Import and reload to get latest code
from src import document_classification
importlib.reload(document_classification)

from src.document_classification import create_classifier, Config, ClassificationMethod

print("✓ Imports successful")

ModuleNotFoundError: No module named 'document_classification'

## Configuration

In [ ]:
# Test document path
test_doc = "/Users/lindamthomas/Documents/GitHub/HSBC_IWPB_UW/experiments/data/sample.pdf"

# Verify file exists
if not Path(test_doc).exists():
    print(f"⚠️  Test file not found: {test_doc}")
    print("Please update the path above")
else:
    print(f"✓ Test file: {Path(test_doc).name}")

## 1. ACU-Only Classification

In [ ]:
print("Running ACU-Only classifier...")
print("-" * 60)

# Set method
Config.CLASSIFICATION_METHOD = ClassificationMethod.ACU_ONLY

# Create classifier
classifier_acu = create_classifier()

# Classify
result_acu = classifier_acu.classify({'path': test_doc})

print(f"✓ Document Type: {result_acu['document_type']}")
print(f"✓ Confidence: {result_acu['confidence']:.3f}")
print(f"✓ Segments: {len(result_acu.get('segments', []))}")
print(f"✓ Tokens: {result_acu['metadata']['token_usage']['total_tokens']:,}")
print("\nSegment details:")
for seg in result_acu.get('segments', []):
    pages = f"p{seg['start_page']}" if seg['start_page'] == seg['end_page'] else f"p{seg['start_page']}-{seg['end_page']}"
    print(f"  • {seg['category']} ({pages})")

## 2. ACU + LLM Text Classification

In [ ]:
print("Running ACU + LLM Text classifier...")
print("-" * 60)

# Set method
Config.CLASSIFICATION_METHOD = ClassificationMethod.ACU_LLM_TEXT

# Create classifier
classifier_text = create_classifier()

# Classify
result_text = classifier_text.classify({'path': test_doc})

print(f"✓ Document Type: {result_text['document_type']}")
print(f"✓ Confidence: {result_text['confidence']:.3f}")
print(f"✓ Pages: {result_text['metadata']['successful_pages']}/{result_text['metadata']['total_pages']}")
print(f"✓ Tokens: {result_text['metadata']['total_tokens']:,}")
print("\nPer-page classifications:")
for page_class in result_text.get('page_classifications', [])[:5]:
    print(f"  • Page {page_class['page']}: {page_class['category']} ({page_class['confidence']:.3f})")

## 3. LLM Image Classification

In [ ]:
print("Running LLM Image classifier...")
print("-" * 60)

# Set method
Config.CLASSIFICATION_METHOD = ClassificationMethod.LLM_IMAGE

# Create classifier
classifier_image = create_classifier()

# Classify
result_image = classifier_image.classify({'path': test_doc})

print(f"✓ Document Type: {result_image['document_type']}")
print(f"✓ Confidence: {result_image['confidence']:.3f}")
print(f"✓ Pages: {result_image['metadata']['successful_pages']}/{result_image['metadata']['total_pages']}")
print(f"✓ Tokens: {result_image['metadata']['total_tokens']:,}")
print("\nPer-page classifications:")
for page_class in result_image.get('page_classifications', [])[:5]:
    print(f"  • Page {page_class['page']}: {page_class['category']} ({page_class['confidence']:.3f})")

## Results Comparison

In [ ]:
# Load results from output folder
output_dir = Path.cwd() / "output"

# Find latest result files for each method
result_files = {
    'acu_only': sorted(output_dir.glob('acu_classification_results_*.json'))[-1] if list(output_dir.glob('acu_classification_results_*.json')) else None,
    'acu_llm_text': sorted(output_dir.glob('acu_llm_text_results_*.json'))[-1] if list(output_dir.glob('acu_llm_text_results_*.json')) else None,
    'llm_image': sorted(output_dir.glob('llm_image_results_*.json'))[-1] if list(output_dir.glob('llm_image_results_*.json')) else None
}

# Load data
results_data = {}
for method, file_path in result_files.items():
    if file_path:
        with open(file_path, 'r') as f:
            results_data[method] = json.load(f)
        print(f"✓ Loaded {method}: {file_path.name}")
    else:
        print(f"⚠️  No results found for {method}")

### Summary Table

In [ ]:
# Create summary dataframe
summary_rows = []

for method, data in results_data.items():
    if isinstance(data, list) and len(data) > 0:
        result = data[0]
    else:
        result = data
    
    metadata = result.get('metadata', {})
    
    # Handle different token usage structures
    if 'total_tokens' in metadata:
        total_tokens = metadata['total_tokens']
    else:
        token_usage = metadata.get('token_usage', {})
        total_tokens = token_usage.get('total_tokens', 0)
    
    # Get page info
    if 'total_pages' in metadata:
        pages_info = f"{metadata.get('successful_pages', 0)}/{metadata['total_pages']}"
    else:
        pages_info = "N/A"
    
    summary_rows.append({
        'Method': method.replace('_', ' ').title(),
        'Document Type': result.get('document_type', 'N/A'),
        'Confidence': f"{result.get('confidence', 0):.3f}",
        'Pages': pages_info,
        'Total Tokens': total_tokens
    })

df_summary = pd.DataFrame(summary_rows)
print("\n" + "="*80)
print("CLASSIFICATION RESULTS SUMMARY")
print("="*80)
display(df_summary)

### Token Usage Comparison

In [ ]:
# Create token comparison chart
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

# Total tokens comparison
methods = df_summary['Method'].tolist()
total_tokens = df_summary['Total Tokens'].tolist()

colors = ['#2ecc71', '#3498db', '#e74c3c']
bars = ax.bar(methods, total_tokens, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Total Tokens', fontsize=12, fontweight='bold')
ax.set_title('Total Token Usage by Method', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Calculate efficiency
baseline = total_tokens[0]
print("\n" + "="*60)
print("EFFICIENCY COMPARISON")
print("="*60)
for i, method in enumerate(methods):
    ratio = total_tokens[i] / baseline
    savings = (1 - ratio) * 100
    if savings > 0:
        print(f"{method:20} : {savings:+.1f}% more efficient than baseline")
    elif savings < 0:
        print(f"{method:20} : {abs(savings):.1f}% less efficient than baseline")
    else:
        print(f"{method:20} : Baseline")

### Confidence Comparison

In [ ]:
# Confidence comparison
fig, ax = plt.subplots(figsize=(10, 6))

confidences = [float(c) for c in df_summary['Confidence']]
bars = ax.barh(methods, confidences, color=colors, alpha=0.7, edgecolor='black')

ax.set_xlabel('Confidence Score', fontsize=12, fontweight='bold')
ax.set_title('Classification Confidence by Method', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.1)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(width + 0.02, bar.get_y() + bar.get_height()/2.,
            f'{confidences[i]:.3f}',
            ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Per-Page Analysis (ACU-Only)

In [ ]:
# Show per-page classification for ACU-only (has segments)
if 'acu_only' in results_data:
    acu_data = results_data['acu_only']
    if isinstance(acu_data, list):
        acu_data = acu_data[0]
    
    segments = acu_data.get('segments', [])
    
    if segments:
        # Create page-level data
        page_data = []
        for seg in segments:
            for page in range(seg['start_page'], seg['end_page'] + 1):
                page_data.append({
                    'Page': page,
                    'Category': seg['category'],
                    'Segment': seg['segment_id']
                })
        
        df_pages = pd.DataFrame(page_data)
        
        print("\n" + "="*60)
        print("PER-PAGE CLASSIFICATION (ACU-Only)")
        print("="*60)
        display(df_pages)
        
        # Visualize
        fig, ax = plt.subplots(figsize=(12, 6))
        
        categories = df_pages['Category'].unique()
        colors_map = {cat: plt.cm.Set3(i) for i, cat in enumerate(categories)}
        
        for _, row in df_pages.iterrows():
            ax.barh(row['Page'], 1, left=0, 
                   color=colors_map[row['Category']], 
                   edgecolor='black', linewidth=2, alpha=0.8)
            ax.text(0.5, row['Page'], row['Category'], 
                   ha='center', va='center', fontweight='bold', fontsize=10)
        
        ax.set_xlabel('', fontsize=12)
        ax.set_ylabel('Page Number', fontsize=12, fontweight='bold')
        ax.set_title('Document Structure - Page-by-Page Classification', 
                    fontsize=14, fontweight='bold')
        ax.set_ylim(0.5, len(df_pages) + 0.5)
        ax.set_xlim(0, 1)
        ax.set_xticks([])
        ax.invert_yaxis()
        
        plt.tight_layout()
        plt.show()

### Page Classifications (ACU + LLM Image)

In [ ]:
# Show per-page classification for text method
if 'acu_llm_text' in results_data:
    text_data = results_data['acu_llm_text']
    if isinstance(text_data, list):
        text_data = text_data[0]
    
    page_classes = text_data.get('page_classifications', [])
    
    if page_classes:
        df_text_pages = pd.DataFrame(page_classes)
        
        print("\n" + "="*60)
        print("PER-PAGE CLASSIFICATION (ACU + LLM Text)")
        print("="*60)
        display(df_text_pages[['page', 'category', 'confidence', 'text_length']])
        
        # Plot confidence by page
        fig, ax = plt.subplots(figsize=(12, 6))
        
        pages = df_text_pages['page'].tolist()
        confidences = df_text_pages['confidence'].tolist()
        categories = df_text_pages['category'].tolist()
        
        # Color by category
        unique_cats = list(set(categories))
        cat_colors = {cat: plt.cm.Set2(i) for i, cat in enumerate(unique_cats)}
        colors_list = [cat_colors[cat] for cat in categories]
        
        bars = ax.bar(pages, confidences, color=colors_list, alpha=0.7, edgecolor='black')
        
        ax.set_xlabel('Page Number', fontsize=12, fontweight='bold')
        ax.set_ylabel('Confidence Score', fontsize=12, fontweight='bold')
        ax.set_title('Classification Confidence per Page (Text Method)', 
                    fontsize=14, fontweight='bold')
        ax.set_ylim(0, 1.1)
        ax.grid(axis='y', alpha=0.3)
        
        # Add category labels on bars
        for i, bar in enumerate(bars):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                   f'{categories[i]}\n{confidences[i]:.3f}',
                   ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        plt.tight_layout()
        plt.show()

### Page Classifications (ACU + LLM Text)

In [ ]:
# Show per-page classification for image method
if 'llm_image' in results_data:
    image_data = results_data['llm_image']
    if isinstance(image_data, list):
        image_data = image_data[0]
    
    page_classes = image_data.get('page_classifications', [])
    
    if page_classes:
        df_image_pages = pd.DataFrame(page_classes)
        
        print("\n" + "="*60)
        print("PER-PAGE CLASSIFICATION (ACU + LLM Image)")
        print("="*60)
        display(df_image_pages)
        
        # Plot confidence by page
        fig, ax = plt.subplots(figsize=(12, 6))
        
        pages = df_image_pages['page'].tolist()
        confidences = df_image_pages['confidence'].tolist()
        categories = df_image_pages['category'].tolist()
        
        # Color by category
        unique_cats = list(set(categories))
        cat_colors = {cat: plt.cm.Set2(i) for i, cat in enumerate(unique_cats)}
        colors_list = [cat_colors[cat] for cat in categories]
        
        bars = ax.bar(pages, confidences, color=colors_list, alpha=0.7, edgecolor='black')
        
        ax.set_xlabel('Page Number', fontsize=12, fontweight='bold')
        ax.set_ylabel('Confidence Score', fontsize=12, fontweight='bold')
        ax.set_title('Classification Confidence per Page (Image Method)', 
                    fontsize=14, fontweight='bold')
        ax.set_ylim(0, 1.1)
        ax.grid(axis='y', alpha=0.3)
        
        # Add category labels on bars
        for i, bar in enumerate(bars):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                   f'{categories[i]}\n{confidences[i]:.3f}',
                   ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        plt.tight_layout()
        plt.show()

## Summary & Recommendations

In [ ]:
print("\n" + "="*80)
print("SUMMARY & RECOMMENDATIONS")
print("="*80)
print()

# Find most efficient method
min_tokens_idx = total_tokens.index(min(total_tokens))
most_efficient = methods[min_tokens_idx]

print(f"✓ All three classification methods successfully tested")
print(f"✓ Document: {Path(test_doc).name}")
print()

print("📊 Token Efficiency Ranking:")
token_ranking = sorted(zip(methods, total_tokens), key=lambda x: x[1])
for i, (method, tokens) in enumerate(token_ranking, 1):
    print(f"   {i}. {method}: {tokens:,} tokens")

print()
print("💡 Recommendations:")
print(f"   • Use {most_efficient} for production (most efficient)")
print("   • Use ACU + LLM Text for custom classification rules")
print("   • Use ACU + LLM Image for documents with complex visual layouts")

print()
print("📁 Results saved to:")
for method, file_path in result_files.items():
    if file_path:
        print(f"   • {file_path.name}")

print("\n" + "="*80)